# NPCore: Simulación de NPCs con comportamiento emergente


Simulación de agentes autónomos capaces de tomar decisiones, interactuar socialmente y navegar en entornos complejos.


Este notebook presenta **NPCore**, una librería en Python para simular agentes autónomos (NPCs) con:

- toma de decisiones basada en reglas
- memoria estructurada
- emociones y personalidad
- aprendizaje adaptativo
- interacción social y comportamiento grupal
- navegación en entornos con obstáculos y costos

---

## Objetivo

El objetivo de este notebook es demostrar cómo construir y analizar sistemas de comportamiento emergente utilizando NPCs que interactúan en un entorno dinámico.

Se mostrará paso a paso:

1. Cómo funciona el sistema de decisiones (`Brain`)
2. Cómo se construyen los agentes (`NPC`)
3. Cómo interactúan dentro de un entorno (`Environment`)
4. Ejemplos progresivos hasta una simulación completa

---

## Enfoque

Este notebook está estructurado como una guía técnica y demostrativa, donde cada componente del sistema se explica y se prueba con ejemplos prácticos.

## Ejemplo integrado: navegación completa

En este ejemplo:

- el NPC tiene una posición inicial
- tiene un destino
- existen obstáculos
- hay zonas costosas

El sistema debe encontrar una ruta adecuada.

## Instalación

Para ejecutar este notebook en Google Colab o localmente, es necesario instalar la librería NPCore.


In [ ]:
%pip install npcore

Note: you may need to restart the kernel to use updated packages.


## Importación de módulos

Importamos las clases principales del sistema:

- `Brain`: motor de decisiones
- `NPC`: agente autónomo
- `Environment`: entorno de simulación

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../src"))

from npcore.brain import Brain
from npcore.npc import NPC
from npcore.environment import Environment

## Arquitectura del sistema

NPCore está basado en una arquitectura modular compuesta por tres elementos principales:

### Brain (Cerebro)
Define cómo un NPC toma decisiones en función de:
- reglas
- contexto
- memoria
- emociones
- aprendizaje

### NPC (Agente)
Representa a un individuo dentro de la simulación. Cada NPC tiene:
- estado
- memoria
- personalidad
- objetivos
- comportamiento social

### Environment (Entorno)
Define el espacio donde los NPCs interactúan:
- mapa con dimensiones
- obstáculos
- zonas con costos
- eventos globales y locales

---

### Flujo general

1. El entorno avanza por pasos (ticks)
2. Cada NPC consulta su `Brain`
3. El `Brain` decide una acción
4. El NPC ejecuta la acción
5. El entorno se actualiza

Este ciclo genera comportamiento emergente a partir de reglas simples.

## Sistema de decisiones: Brain

El `Brain` es el componente encargado de decidir qué acción realiza un NPC.

Funciona mediante un sistema de reglas:

- Cada estado tiene una función asociada
- La función devuelve posibles acciones con pesos
- El sistema selecciona una acción probabilísticamente

Esto permite modelar comportamiento flexible y adaptable.

### Función: add_rule

La función `add_rule` permite definir cómo se comporta un NPC en un estado específico.

Recibe:
- el nombre del estado
- una función que describe el comportamiento

Esta función de comportamiento recibe un contexto y devuelve un diccionario de acciones con pesos.

Ejemplo:
- "run": 1.0
- "wait": 0.5

Esto indica qué tan probable es cada acción.

In [ ]:
brain = Brain()

def idle_rule(context):
    return {
        "run": 1.0,
        "wait": 1.0
    }

brain.add_rule("idle", idle_rule)

### Función: decide

La función `decide` es utilizada internamente por el NPC para elegir una acción.

Proceso:

1. Se obtiene la regla asociada al estado actual
2. Se calculan los pesos de cada acción
3. Se selecciona una acción de manera probabilística

Este enfoque permite:

- comportamiento no determinista
- variabilidad entre ejecuciones
- simulación más realista

In [ ]:
# Simulación simple de decisión

context = {}

for _ in range(5):
    decision = brain.decide("idle", context)
    print("Decisión:", decision)

Decisión: run
Decisión: wait
Decisión: wait
Decisión: run
Decisión: wait


## Ejemplo integrado: Brain + NPC

A continuación, se muestra cómo el Brain se conecta con un NPC dentro de un entorno.

In [ ]:
npc = NPC("Guard", brain)
npc.set_state("idle")

env = Environment()

env.add_npc(npc)

env.run(steps=3)

print(env.summary())

{'ticks': 3, 'npcs': 1, 'history_length': 3, 'action_counts': {'wait': 2, 'run': 1}}


## Sistema de agentes: NPC

Un NPC (Non-Player Character) representa un agente autónomo dentro de la simulación.

Cada NPC cuenta con:

- un estado interno
- memoria
- emociones
- rasgos de personalidad
- capacidad de aprendizaje
- interacción con otros agentes

El NPC utiliza su `Brain` para decidir acciones, pero su comportamiento está influenciado por su estado interno.

### Estado del NPC: set_state

El estado define el contexto actual del NPC.

Ejemplos de estados:
- "idle"
- "explore"
- "flee"
- "follow"

El estado determina qué regla del `Brain` se utilizará para tomar decisiones.

In [ ]:
brain = Brain()

def idle_rule(context):
    return {"wait": 1.0}

brain.add_rule("idle", idle_rule)

npc = NPC("Scout", brain)

npc.set_state("idle")

env = Environment()
env.add_npc(npc)

env.run(steps=2)

print(env.summary())

{'ticks': 2, 'npcs': 1, 'history_length': 2, 'action_counts': {'wait': 2}}


### Contexto del NPC: update_context

El contexto permite pasar información al sistema de decisiones.

Ejemplos:
- peligro cercano
- presencia de aliados
- eventos recientes

El `Brain` puede usar este contexto para modificar decisiones.

In [ ]:
def danger_rule(context):
    if context.get("danger"):
        return {"run": 2.0, "hide": 1.0}
    return {"wait": 1.0}

brain.add_rule("alert", danger_rule)

npc.set_state("alert")

npc.update_context(danger=True)

env.run(steps=2)

print(env.summary())

{'ticks': 4, 'npcs': 1, 'history_length': 4, 'action_counts': {'wait': 2, 'run': 1, 'hide': 1}}


### Memoria básica: remember, recall, forget

Los NPCs pueden almacenar información en memoria.

Funciones:
- `remember(key, value)`
- `recall(key)`
- `forget(key)`

Esto permite que los agentes mantengan información entre decisiones.

In [ ]:
npc.remember("last_position", (2, 3))

print("Memoria:", npc.recall("last_position"))

npc.forget("last_position")

print("Después de olvidar:", npc.recall("last_position"))

Memoria: (2, 3)
Después de olvidar: None


### Memoria de eventos

Los NPCs pueden registrar eventos ocurridos durante la simulación.

Funciones:
- `remember_event(tipo, datos)`
- `recall_last_event()`
- `recall_events_by_type(tipo)`

Esto permite analizar el comportamiento a lo largo del tiempo.

In [ ]:
npc.remember_event("danger", {"level": "high"})
npc.remember_event("ally_seen", {"name": "Guard"})

print("Último evento:", npc.recall_last_event())

print("Eventos de peligro:", npc.recall_events_by_type("danger"))

Último evento: {'type': 'ally_seen', 'source': {'name': 'Guard'}, 'target': None, 'detail': None, 'priority': 1}
Eventos de peligro: [{'type': 'danger', 'source': {'level': 'high'}, 'target': None, 'detail': None, 'priority': 1}]


### Emociones

Los NPCs pueden tener estados emocionales que afectan su comportamiento.

Funciones:
- `set_emotion(nombre, valor)`
- `get_emotion(nombre)`

Ejemplo:
- miedo alto → más probabilidad de huir
- confianza alta → más probabilidad de explorar

In [ ]:
npc.set_emotion("fear", 0.9)

print("Nivel de miedo:", npc.get_emotion("fear"))

Nivel de miedo: 0.9


### Personalidad

Los NPCs pueden tener rasgos de personalidad persistentes.

Funciones:
- `set_personality_trait(nombre, valor)`
- `get_personality_trait(nombre)`

Esto define diferencias entre agentes incluso en el mismo entorno.

In [ ]:
npc.set_personality_trait("aggressiveness", 0.8)

print("Agresividad:", npc.get_personality_trait("aggressiveness"))

Agresividad: 0.8


### Aprendizaje

Los NPCs pueden aprender de experiencias pasadas.

Funciones:
- `record_outcome(action, success)`
- `get_action_success_rate(action)`
- `get_learning_weight(action)`

Esto permite adaptar el comportamiento con el tiempo.

In [ ]:
npc.record_outcome("run", True)
npc.record_outcome("run", False)
npc.record_outcome("run", True)

print("Historial:", npc.get_action_history("run"))

print("Tasa de éxito:", npc.get_action_success_rate("run"))

print("Peso aprendido:", npc.get_learning_weight("run"))

Historial: [True, False, True]
Tasa de éxito: 0.6666666666666666
Peso aprendido: 1.6666666666666665


## Sistema de memoria del NPC

Los NPCs en NPCore pueden almacenar y utilizar información de eventos pasados.

Esto permite simular:

- aprendizaje basado en experiencia
- toma de decisiones informada
- comportamiento más realista

La memoria funciona como un registro de eventos organizados por tipo.

### Función: remember_event

Permite guardar un evento en la memoria del NPC.

Parámetros:

- `event`: nombre del evento (string)
- `data`: información asociada al evento

Esto permite registrar información relevante del entorno.   

In [ ]:
npc.remember_event("danger", {"level": 3, "source": "enemy"})
npc.remember_event("danger", {"level": 5, "source": "trap"})
npc.remember_event("ally_seen", {"name": "Scout"})

### Función: recall

Permite recuperar los eventos almacenados de un tipo específico.

Esto es útil para que el NPC tome decisiones basadas en experiencias pasadas.

In [ ]:
danger_events = npc.recall("danger")
print("Eventos de peligro:", danger_events)

ally_events = npc.recall("ally_seen")
print("Aliados vistos:", ally_events)

Eventos de peligro: None
Aliados vistos: None


### Función: forget

Permite eliminar eventos almacenados en la memoria.

Esto puede representar:

- olvido
- limpieza de memoria
- cambios de contexto

In [ ]:
npc.forget("danger")

print("Memoria después de olvidar peligro:", npc.recall("danger"))

Memoria después de olvidar peligro: None


## Ejemplo integrado de memoria

A continuación, se muestra un ejemplo completo donde el NPC:

1. Guarda eventos
2. Consulta su memoria
3. Elimina información

Esto permite observar el comportamiento dinámico del sistema de memoria.

In [ ]:
# Guardar eventos
npc.remember_event("danger", {"level": 2})
npc.remember_event("danger", {"level": 4})
npc.remember_event("ally_seen", {"name": "Captain"})

# Consultar memoria
print("Peligros registrados:", npc.recall("danger"))
print("Aliados vistos:", npc.recall("ally_seen"))

# Olvidar eventos
npc.forget("danger")

print("Memoria final:", npc.recall("danger"))

Peligros registrados: None
Aliados vistos: None
Memoria final: None


## Interpretación

El sistema de memoria permite que los NPCs:

- recuerden eventos pasados
- adapten su comportamiento
- simulen aprendizaje y experiencia

Esto es clave para generar comportamiento emergente en sistemas multi-agente.

## Emociones, personalidad y contexto social

Además de memoria y aprendizaje, los NPCs en NPCore pueden modificar su comportamiento con base en:

- emociones
- rasgos de personalidad
- contexto social

Esto permite construir agentes que no solo reaccionan a reglas fijas, sino que muestran diferencias individuales y respuestas más realistas.

### Función: set_emotion y get_emotion

Las emociones representan estados internos temporales del NPC.

Ejemplos:
- miedo
- confianza
- agresión

Estas emociones pueden modificar la forma en que el NPC toma decisiones.

In [ ]:
npc.set_emotion("fear", 0.9)
npc.set_emotion("trust", 0.2)

print("Miedo:", npc.get_emotion("fear"))
print("Confianza:", npc.get_emotion("trust"))

Miedo: 0.9
Confianza: 0.2


### Función: set_personality_trait y get_personality_trait

La personalidad representa rasgos más estables del NPC.

Ejemplos:
- aggression
- sociability
- fearfulness
- loyalty

A diferencia de las emociones, estos rasgos no cambian constantemente y ayudan a diferenciar agentes.

In [ ]:
npc.set_personality_trait("aggression", 0.8)
npc.set_personality_trait("loyalty", 0.9)

print("Agresión:", npc.get_personality_trait("aggression"))
print("Lealtad:", npc.get_personality_trait("loyalty"))

Agresión: 0.8
Lealtad: 0.9


### Emociones en la toma de decisiones

A continuación se muestra un ejemplo donde el miedo modifica el comportamiento del NPC.

La idea es que un NPC con mucho miedo tenga más probabilidad de huir.

In [ ]:
brain_fear = Brain()

def fear_rule(context):
    return {"run": 1.0, "wait": 1.0}

brain_fear.add_rule("react", fear_rule)

fearful_npc = NPC("Guard", brain_fear)
fearful_npc.set_state("react")
fearful_npc.set_emotion("fear", 1.0)

results = [fearful_npc.act() for _ in range(20)]
print(results)
print("Veces que corrió:", results.count("run"))
print("Veces que esperó:", results.count("wait"))

['wait', 'run', 'wait', 'run', 'run', 'run', 'run', 'run', 'wait', 'wait', 'run', 'run', 'run', 'run', 'run', 'run', 'run', 'wait', 'run', 'run']
Veces que corrió: 15
Veces que esperó: 5


### Personalidad en la toma de decisiones

En este ejemplo, un rasgo de personalidad estable influye en la decisión del NPC.

La agresividad alta debería favorecer acciones más activas o directas.

In [ ]:
brain_personality = Brain()

def personality_rule(context):
    return {"attack": 1.0, "wait": 1.0}

brain_personality.add_rule("combat", personality_rule)

aggressive_npc = NPC("Raider", brain_personality)
aggressive_npc.set_state("combat")
aggressive_npc.set_personality_trait("aggression", 1.0)

results = [aggressive_npc.act() for _ in range(20)]
print(results)
print("Veces que atacó:", results.count("attack"))
print("Veces que esperó:", results.count("wait"))

['wait', 'wait', 'attack', 'attack', 'wait', 'attack', 'attack', 'wait', 'attack', 'attack', 'attack', 'attack', 'attack', 'attack', 'attack', 'attack', 'attack', 'attack', 'attack', 'attack']
Veces que atacó: 16
Veces que esperó: 4


### Función: set_group y set_rank

Los NPCs también pueden formar parte de una estructura social.

Esto permite modelar:

- grupos
- aliados
- líderes
- jerarquías

Estas estructuras son útiles para simular coordinación y comportamiento grupal.

In [ ]:
captain = NPC("Captain", brain)
guard = NPC("Guard", brain)

captain.set_group("guards")
guard.set_group("guards")

captain.set_rank("leader")
guard.set_rank("soldier")

print("Grupo del Captain:", captain.group)
print("Rango del Captain:", captain.rank)
print("Grupo del Guard:", guard.group)
print("Rango del Guard:", guard.rank)

Grupo del Captain: guards
Rango del Captain: leader
Grupo del Guard: guards
Rango del Guard: soldier


### Función: set_relationship, change_relationship y get_relationship

Los NPCs pueden almacenar relaciones con otros agentes.

Esto permite representar:
- confianza
- rivalidad
- cooperación
- afinidad social

Las relaciones pueden cambiar con el tiempo.

In [ ]:
guard.set_relationship("Captain", 0.5)
print("Relación inicial con Captain:", guard.get_relationship("Captain"))

guard.change_relationship("Captain", 0.3)
print("Relación actualizada con Captain:", guard.get_relationship("Captain"))

Relación inicial con Captain: 0.5
Relación actualizada con Captain: 0.8


### Función: get_social_influence

Esta función resume cómo influyen otros NPCs cercanos sobre el agente actual.

Puede contar:
- líderes
- aliados
- otros agentes

Esto ayuda a modelar presión de grupo o coordinación social.

In [ ]:
ally_1 = NPC("Scout", brain)
ally_2 = NPC("Medic", brain)
enemy = NPC("Bandit", brain)

guard.set_group("guards")
captain.set_group("guards")
captain.set_rank("leader")
ally_1.set_group("guards")
ally_2.set_group("guards")
enemy.set_group("bandits")

influence = guard.get_social_influence([captain, ally_1, ally_2, enemy])
print(influence)

{'leaders': 1, 'allies': 3, 'others': 1}


### Ejemplo integrado: emoción, personalidad y entorno social

En este ejemplo combinamos tres dimensiones del NPC:

- emoción: miedo
- personalidad: lealtad
- contexto social: presencia de un líder

Esto permite mostrar que el comportamiento del agente no depende de una sola variable, sino de la interacción entre varios factores internos y sociales.

In [ ]:
brain_social = Brain()

def social_rule(context):
    return {"run": 1.0, "follow": 1.0}

brain_social.add_rule("conflict", social_rule)

leader = NPC("Captain", brain_social)
leader.set_group("guards")
leader.set_rank("leader")
leader.set_position(2, 0)

soldier = NPC("Guard", brain_social)
soldier.set_state("conflict")
soldier.set_group("guards")
soldier.set_position(0, 0)
soldier.set_emotion("fear", 1.0)
soldier.set_personality_trait("loyalty", 1.0)

soldier.update_context(nearby=[leader])

results = [soldier.act() for _ in range(30)]
print(results)
print("Run:", results.count("run"))
print("Follow:", results.count("follow"))

['follow', 'run', 'run', 'run', 'follow', 'follow', 'follow', 'run', 'follow', 'run', 'run', 'follow', 'follow', 'follow', 'run', 'follow', 'run', 'run', 'run', 'follow', 'follow', 'run', 'run', 'run', 'follow', 'follow', 'run', 'run', 'run', 'run']
Run: 17
Follow: 13


## Interpretación

Este bloque muestra que los NPCs de NPCore no son agentes estáticos.

Cada uno puede diferir por:

- emociones momentáneas
- rasgos persistentes de personalidad
- relaciones con otros agentes
- posición dentro de un grupo

Esto hace posible construir simulaciones más complejas y expresivas.

## Movimiento y navegación en el entorno

Los NPCs en NPCore pueden moverse dentro de un entorno que incluye:

- posiciones en un mapa
- obstáculos
- zonas con diferentes costos
- destinos

Esto permite simular navegación inteligente y toma de decisiones espaciales.

### Función: set_position

Permite asignar una posición inicial a un NPC dentro del entorno.

Parámetros:
- x: coordenada horizontal
- y: coordenada vertical

In [ ]:
npc.set_position(0, 0)
print("Posición inicial:", npc.position)

Posición inicial: (0, 0)


### Función: set_destination

Define el objetivo hacia el cual el NPC intentará moverse.

Esto permite modelar comportamiento dirigido a metas.

In [ ]:
npc.set_destination(4, 2)
print("Destino:", npc.destination)

Destino: (4, 2)


### Obstáculos en el entorno

El entorno puede contener celdas bloqueadas que los NPCs no pueden atravesar.

Esto obliga al sistema a encontrar rutas alternativas.

In [ ]:
env.add_block(2, 1)
env.add_block(2, 2)

print("Celdas bloqueadas:", env.blocked_cells)
print("¿La celda (2, 1) está bloqueada?", env.is_blocked(2, 1))

Celdas bloqueadas: {(2, 1), (2, 2)}
¿La celda (2, 1) está bloqueada? True


### Costo de movimiento por celda

Algunas celdas del entorno pueden tener un costo mayor para el movimiento.

Esto permite modelar:

- zonas peligrosas
- terreno difícil
- áreas a evitar

En lugar de solo bloquear el paso, el sistema puede hacer que ciertas rutas sean más caras.

In [ ]:
env.set_cell_cost(1, 1, 3)
env.set_cell_cost(1, 2, 5)

print("Costo en (1, 1):", env.get_cell_cost(1, 1))
print("Costo en (1, 2):", env.get_cell_cost(1, 2))

Costo en (1, 1): 3
Costo en (1, 2): 5


### Zonas con costo

Algunas áreas del entorno pueden tener un costo mayor para el movimiento.

Esto permite modelar:
- zonas peligrosas
- terreno difícil
- áreas a evitar

In [ ]:
env.add_zone((1, 1), 3)
env.add_zone((1, 2), 5)

print("Zonas con costo:", env.zones)

Zonas con costo: {(1, 1): 3, (1, 2): 5}


### Ejecución del entorno

El entorno avanza en pasos (ticks).

En cada paso:
1. Cada NPC toma una decisión
2. Se actualiza su posición
3. Se registra la acción

Esto permite observar la evolución del sistema.

In [ ]:
env.run(steps=5)

print(env.summary())

{'ticks': 9, 'npcs': 1, 'history_length': 9, 'action_counts': {'wait': 2, 'run': 5, 'hide': 2}}


### Visualización del entorno

El entorno puede representarse como una cuadrícula donde:

- cada NPC ocupa una celda
- los obstáculos bloquean el paso
- el destino es visible
- se puede observar la evolución

Esto permite analizar el comportamiento espacial.

In [ ]:
print(env.render_grid(5, 3))

    0   1   2   3   4
  +---+---+---+---+---+
0 | S | . | . | . | . |
  +---+---+---+---+---+
1 | . | . | # | . | . |
  +---+---+---+---+---+
2 | . | . | # | . | D |
  +---+---+---+---+---+

Legend:
S = Scout
# = blocked cell
D = destination


### Evaluación de riesgo

Los NPCs pueden evaluar el riesgo en su entorno antes de moverse.

Esto depende de factores como:

- zonas peligrosas
- presencia de enemigos
- condiciones del entorno

Esto influye directamente en sus decisiones.

In [ ]:
risk = npc.assess_local_risk(env)
print("Nivel de riesgo:", risk)

Nivel de riesgo: 6


## Ejemplo integrado: navegación completa

En este ejemplo:

- el NPC tiene una posición inicial
- tiene un destino
- existen obstáculos
- hay zonas costosas

El sistema debe encontrar una ruta adecuada.

In [ ]:
brain_nav = Brain()

def nav_rule(context):
    return {"run": 1.0}

brain_nav.add_rule("idle", nav_rule)

env2 = Environment()

npc2 = NPC("Explorer", brain_nav)
npc2.set_state("idle")
npc2.set_position(0, 0)
npc2.set_destination(4, 2)

# Obstáculos
env2.add_block(2, 0)
env2.add_block(2, 1)
env2.add_block(2, 2)

# Celdas costosas
env2.set_cell_cost(1, 1, 3)
env2.set_cell_cost(3, 1, 4)

env2.add_npc(npc2)

env2.run(steps=10)

print(env2.render_grid(5, 3))
print(env2.summary())
print("Riesgo local final:", npc2.assess_local_risk(env2))

    0   1   2   3   4
  +---+---+---+---+---+
0 | E | . | # | . | . |
  +---+---+---+---+---+
1 | . | . | # | . | . |
  +---+---+---+---+---+
2 | . | . | # | . | D |
  +---+---+---+---+---+

Legend:
E = Explorer
# = blocked cell
D = destination
{'ticks': 10, 'npcs': 1, 'history_length': 10, 'action_counts': {'run': 10}}
Riesgo local final: 6


## Interpretación

Este bloque demuestra que los NPCs pueden:

- navegar en un entorno complejo
- evitar obstáculos
- considerar costos de movimiento
- tomar decisiones espaciales

Esto permite modelar comportamiento autónomo en sistemas dinámicos.

Es una base para simulaciones más avanzadas como:

- videojuegos
- sistemas multi-agente
- inteligencia artificial distribuida

## Comportamiento social y coordinación grupal

Además del comportamiento individual, NPCore permite modelar interacción entre múltiples NPCs.

Esto incluye:

- formación de grupos
- jerarquías (rangos)
- relaciones entre agentes
- influencia social
- coordinación en el entorno

Esto permite simular sistemas multiagente más complejos y realistas.

In [ ]:
brain_group = Brain()

def idle_rule(context):
    return {"wait": 1.0}

brain_group.add_rule("idle", idle_rule)

env_social = Environment()

leader = NPC("Captain", brain_group)
ally1 = NPC("Guard1", brain_group)
ally2 = NPC("Guard2", brain_group)

leader.set_state("idle")
ally1.set_state("idle")
ally2.set_state("idle")

leader.set_position(0, 0)
ally1.set_position(1, 0)
ally2.set_position(2, 0)

# Grupo y jerarquía
leader.set_group("team_alpha")
ally1.set_group("team_alpha")
ally2.set_group("team_alpha")

leader.set_rank(2)   # líder
ally1.set_rank(1)
ally2.set_rank(1)

env_social.add_npc(leader)
env_social.add_npc(ally1)
env_social.add_npc(ally2)

print("Grupo de leader:", leader.group)
print("Rango de leader:", leader.rank)

Grupo de leader: team_alpha
Rango de leader: 2


### Relaciones entre NPCs

Los NPCs pueden tener relaciones entre ellos, lo que influye en su comportamiento.

Se pueden definir como:

- aliado
- neutral
- enemigo

Estas relaciones afectan decisiones futuras y coordinación.

In [ ]:
ally1.set_relationship("Captain", "ally")
ally2.set_relationship("Captain", "ally")

print("Relación Guard1 -> Captain:", ally1.get_relationship("Captain"))

# Cambiar relación
ally1.change_relationship("Captain", "neutral")

print("Relación actualizada Guard1 -> Captain:", ally1.get_relationship("Captain"))

Relación Guard1 -> Captain: ally
Relación actualizada Guard1 -> Captain: allyneutral


### Influencia social

Cada NPC puede evaluar su entorno social, considerando:

- aliados cercanos
- líderes
- NPCs externos

Esto permite tomar decisiones basadas en el contexto social.

In [ ]:
influence = leader.get_social_influence(env_social.npcs)

print("Influencia social del líder:", influence)

Influencia social del líder: {'leaders': 0, 'allies': 3, 'others': 0}


### Coordinación entre NPCs

Los NPCs pueden coordinar acciones dentro de un grupo:

- compartir eventos
- reaccionar a amenazas
- sincronizar comportamiento

Esto permite comportamiento emergente en simulaciones multiagente.

In [ ]:
env_social2 = Environment()

leader2 = NPC("Captain", brain_group)
guardA = NPC("GuardA", brain_group)
guardB = NPC("GuardB", brain_group)

leader2.set_state("idle")
guardA.set_state("idle")
guardB.set_state("idle")

leader2.set_position(0, 0)
guardA.set_position(1, 0)
guardB.set_position(2, 0)

leader2.set_group("squad")
guardA.set_group("squad")
guardB.set_group("squad")

leader2.set_rank(2)
guardA.set_rank(1)
guardB.set_rank(1)

# Evento en el entorno (simulación simple)
event = {"type": "danger", "position": (2, 2)}

# Agregar NPCs
env_social2.add_npc(leader2)
env_social2.add_npc(guardA)
env_social2.add_npc(guardB)

env_social2.run(steps=5)

print(env_social2.render_grid(5, 3))
print(env_social2.summary())

    0   1   2   3   4
  +---+---+---+---+---+
0 | C | G | G | . | . |
  +---+---+---+---+---+
1 | . | . | . | . | . |
  +---+---+---+---+---+
2 | . | . | . | . | . |
  +---+---+---+---+---+

Legend:
C = Captain
G = GuardB
{'ticks': 5, 'npcs': 3, 'history_length': 5, 'action_counts': {'wait': 15}}


## Demo final: simulación completa

En este bloque se integra todo lo construido en NPCore.

La simulación final reúne:

- toma de decisiones con `Brain`
- múltiples NPCs
- estados internos
- emociones y personalidad
- relaciones sociales
- comportamiento grupal
- navegación en el entorno
- visualización del mapa
- resumen estadístico
- narrativa final

Este ejemplo funciona como una demostración completa del sistema.

In [ ]:
# Brain principal para la demo final
brain_final = Brain()

def idle_rule(context):
    if context.get("danger"):
        return {"run": 2.0, "hide": 1.0, "follow": 1.0}
    return {"wait": 1.0, "run": 1.0}

brain_final.add_rule("idle", idle_rule)

# Entorno final
env_final = Environment()

# Crear NPCs
captain = NPC("Captain", brain_final)
guard = NPC("Guard", brain_final)
scout = NPC("Scout", brain_final)
villager = NPC("Villager", brain_final)

# Estados
captain.set_state("idle")
guard.set_state("idle")
scout.set_state("idle")
villager.set_state("idle")

# Posiciones iniciales
captain.set_position(1, 1)
guard.set_position(1, 2)
scout.set_position(0, 1)
villager.set_position(5, 1)

# Destino compartido
captain.set_destination(6, 4)
guard.set_destination(6, 4)
scout.set_destination(6, 4)
villager.set_destination(6, 4)

# Grupo y jerarquía
captain.set_group("rescue")
guard.set_group("rescue")
scout.set_group("rescue")

captain.set_rank(2)
guard.set_rank(1)
scout.set_rank(1)

# Relaciones
guard.set_relationship("Captain", "ally")
scout.set_relationship("Captain", "ally")
captain.set_relationship("Guard", "ally")
captain.set_relationship("Scout", "ally")

# Emociones y personalidad
guard.set_emotion("fear", 0.6)
scout.set_emotion("fear", 0.2)
villager.set_emotion("fear", 0.9)

guard.set_personality_trait("loyalty", 0.9)
scout.set_personality_trait("curiosity", 0.8)
villager.set_personality_trait("fearfulness", 1.0)

# Contexto
captain.update_context(danger=True)
guard.update_context(danger=True)
scout.update_context(danger=True)
villager.update_context(danger=True)

# Obstáculos
env_final.add_block(3, 1)
env_final.add_block(3, 3)
env_final.add_block(4, 3)

# Costos de movimiento
env_final.set_cell_cost(2, 2, 3)
env_final.set_cell_cost(5, 2, 4)

# Agregar NPCs al entorno
env_final.add_npc(captain)
env_final.add_npc(guard)
env_final.add_npc(scout)
env_final.add_npc(villager)

# Mostrar mundo inicial
print("=== Mundo inicial ===")
print(env_final.render_grid(8, 6))

# Ejecutar simulación
env_final.run(steps=5)

# Mostrar mundo final
print("\n=== Mundo final ===")
print(env_final.render_grid(8, 6))

# Resumen
print("\n=== Summary ===")
print(env_final.summary())

# Narrativa
print("\n=== History ===")
for step in env_final.history:
    print(step)

=== Mundo inicial ===
    0   1   2   3   4   5   6   7
  +---+---+---+---+---+---+---+---+
0 | . | . | . | . | . | . | . | . |
  +---+---+---+---+---+---+---+---+
1 | S | C | . | # | . | V | . | . |
  +---+---+---+---+---+---+---+---+
2 | . | G | . | . | . | . | . | . |
  +---+---+---+---+---+---+---+---+
3 | . | . | . | # | # | . | . | . |
  +---+---+---+---+---+---+---+---+
4 | . | . | . | . | . | . | D | . |
  +---+---+---+---+---+---+---+---+
5 | . | . | . | . | . | . | . | . |
  +---+---+---+---+---+---+---+---+

Legend:
C = Captain
G = Guard
S = Scout
V = Villager
# = blocked cell
D = destination

=== Mundo final ===
    0   1   2   3   4   5   6   7
  +---+---+---+---+---+---+---+---+
0 | . | . | . | . | . | . | . | . |
  +---+---+---+---+---+---+---+---+
1 | S | C | . | # | . | V | . | . |
  +---+---+---+---+---+---+---+---+
2 | . | G | . | . | . | . | . | . |
  +---+---+---+---+---+---+---+---+
3 | . | . | . | # | # | . | . | . |
  +---+---+---+---+---+---+---+---+
4 | . | . 

## Interpretación

La simulación final muestra que NPCore puede integrar en una sola corrida:

- comportamiento individual
- interacción social
- adaptación al contexto
- navegación espacial
- visualización del entorno
- generación de narrativa

Esto convierte a la librería en una base sólida para:

- simulaciones multiagente
- prototipos de IA para videojuegos
- experimentación con comportamiento emergente
- proyectos de inteligencia artificial explicable

## Pruebas y robustez

Para validar el comportamiento del sistema, se implementaron múltiples pruebas que verifican:

- movimiento de NPCs
- asignación de estados
- evaluación de riesgo
- interacción social
- navegación en el entorno
- manejo de obstáculos
- costos de movimiento

Estas pruebas permiten garantizar que el sistema funciona correctamente en distintos escenarios.

Además, ayudan a detectar errores, validar reglas y asegurar consistencia en la simulación.

In [ ]:
print("=== INICIANDO PRUEBAS ===")

# Entorno de prueba
test_env = Environment()

# Brain simple
test_brain = Brain()

def test_rule(context):
    return {"run": 1.0}

test_brain.add_rule("idle", test_rule)

# Crear NPC
test_npc = NPC("Tester", test_brain)
test_npc.set_state("idle")
test_npc.set_position(0, 0)
test_npc.set_destination(2, 2)

# Agregar al entorno
test_env.add_npc(test_npc)

# Test 1: movimiento
test_env.run(steps=1)
print("✔ Movimiento ejecutado")

# Test 2: obstáculos
test_env.add_block(1, 1)
print("✔ Obstáculo agregado:", test_env.is_blocked(1, 1))

# Test 3: costo de celda
test_env.set_cell_cost(0, 1, 5)
print("✔ Costo asignado:", test_env.get_cell_cost(0, 1))

# Test 4: evaluación de riesgo
risk = test_npc.assess_local_risk(test_env)
print("✔ Riesgo calculado:", risk)

# Test 5: relaciones sociales
npc2 = NPC("Ally", test_brain)
npc2.set_group("team")
test_npc.set_group("team")

test_env.add_npc(npc2)

social = test_npc.get_social_influence(test_env.npcs)
print("✔ Influencia social:", social)

# Test 6: render del entorno
print("✔ Render:")
print(test_env.render_grid(4, 4))

# Test 7: summary
print("✔ Summary:")
print(test_env.summary())

print("\n=== TODAS LAS PRUEBAS EJECUTADAS ===")

=== INICIANDO PRUEBAS ===
✔ Movimiento ejecutado
✔ Obstáculo agregado: True
✔ Costo asignado: 5
✔ Riesgo calculado: 8
✔ Influencia social: {'leaders': 0, 'allies': 2, 'others': 0}
✔ Render:
    0   1   2   3
  +---+---+---+---+
0 | T | . | . | . |
  +---+---+---+---+
1 | . | # | . | . |
  +---+---+---+---+
2 | . | . | D | . |
  +---+---+---+---+
3 | . | . | . | . |
  +---+---+---+---+

Legend:
T = Tester
# = blocked cell
D = destination
✔ Summary:
{'ticks': 1, 'npcs': 2, 'history_length': 1, 'action_counts': {'run': 1}}

=== TODAS LAS PRUEBAS EJECUTADAS ===


## Interpretación

Las pruebas realizadas confirman que el sistema:

- responde correctamente a estímulos del entorno
- maneja múltiples NPCs sin errores
- integra comportamiento individual y social
- permite modificar dinámicamente el entorno
- mantiene consistencia en la simulación

El uso de pruebas facilita la extensión del sistema y asegura su estabilidad en escenarios más complejos.

### Escenario grande: navegación en mapa complejo

A continuación se presenta una simulación en un mapa más amplio, con múltiples obstáculos y zonas de costo.

Este ejemplo permite mostrar que el sistema puede manejar:

- mapas de mayor tamaño
- rutas con bloqueos complejos
- costos de movimiento distribuidos
- múltiples NPCs moviéndose en el mismo entorno
- evaluación de riesgo en escenarios más ricos

Este tipo de ejemplo permite visualizar mejor la potencia del sistema de navegación.

In [ ]:
print("=== ESCENARIO GRANDE: NAVEGACIÓN EN MAPA COMPLEJO ===")

brain_big = Brain()

def nav_rule(context):
    if context.get("danger"):
        return {"run": 2.0, "hide": 1.0}
    return {"run": 1.0, "wait": 0.2}

brain_big.add_rule("idle", nav_rule)

env_big = Environment()

# Crear NPCs
captain_big = NPC("Captain", brain_big)
guard_big = NPC("Guard", brain_big)
scout_big = NPC("Scout", brain_big)
villager_big = NPC("Villager", brain_big)

# Estado
for npc in [captain_big, guard_big, scout_big, villager_big]:
    npc.set_state("idle")
    npc.update_context(danger=True)

# Posiciones iniciales
captain_big.set_position(0, 1)
guard_big.set_position(0, 3)
scout_big.set_position(1, 0)
villager_big.set_position(2, 5)

# Destino común
captain_big.set_destination(11, 6)
guard_big.set_destination(11, 6)
scout_big.set_destination(11, 6)
villager_big.set_destination(11, 6)

# Grupo
captain_big.set_group("evac")
guard_big.set_group("evac")
scout_big.set_group("evac")

captain_big.set_rank(2)
guard_big.set_rank(1)
scout_big.set_rank(1)

# Relaciones
guard_big.set_relationship("Captain", "ally")
scout_big.set_relationship("Captain", "ally")
captain_big.set_relationship("Guard", "ally")
captain_big.set_relationship("Scout", "ally")

# Emociones / personalidad
guard_big.set_emotion("fear", 0.5)
scout_big.set_emotion("fear", 0.2)
villager_big.set_emotion("fear", 1.0)

guard_big.set_personality_trait("loyalty", 0.9)
scout_big.set_personality_trait("curiosity", 0.8)
villager_big.set_personality_trait("fearfulness", 1.0)

# Obstáculos tipo muro / laberinto
blocked_cells = [
    (3,0), (3,1), (3,2), (3,3),
    (6,1), (6,2), (6,3), (6,4),
    (8,0), (8,1), (8,2),
    (9,4), (9,5),
    (4,5), (5,5), (6,5),
]

for x, y in blocked_cells:
    env_big.add_block(x, y)

# Celdas costosas
cost_cells = {
    (1,2): 3,
    (2,2): 4,
    (4,1): 3,
    (5,1): 4,
    (7,4): 5,
    (10,3): 3,
    (10,4): 4,
}

for (x, y), cost in cost_cells.items():
    env_big.set_cell_cost(x, y, cost)

# Agregar NPCs
for npc in [captain_big, guard_big, scout_big, villager_big]:
    env_big.add_npc(npc)

# Mundo inicial
print("\n=== MAPA INICIAL ===")
print(env_big.render_grid(12, 7))

# Simulación
env_big.run(steps=12)

# Mundo final
print("\n=== MAPA FINAL ===")
print(env_big.render_grid(12, 7))

# Summary
print("\n=== SUMMARY ===")
print(env_big.summary())

# Riesgo local de cada NPC
print("\n=== RIESGO LOCAL FINAL ===")
for npc in [captain_big, guard_big, scout_big, villager_big]:
    print(f"{npc.name}: posición={npc.position}, riesgo={npc.assess_local_risk(env_big)}")

# Historial
print("\n=== HISTORY ===")
for i, step in enumerate(env_big.history, start=1):
    print(f"Paso {i}: {step}")

=== ESCENARIO GRANDE: NAVEGACIÓN EN MAPA COMPLEJO ===

=== MAPA INICIAL ===
    0   1   2   3   4   5   6   7   8   9   10   11
  +---+---+---+---+---+---+---+---+---+---+---+---+
0 | . | S | . | # | . | . | . | . | # | . | . | . |
  +---+---+---+---+---+---+---+---+---+---+---+---+
1 | C | . | . | # | . | . | # | . | # | . | . | . |
  +---+---+---+---+---+---+---+---+---+---+---+---+
2 | . | . | . | # | . | . | # | . | # | . | . | . |
  +---+---+---+---+---+---+---+---+---+---+---+---+
3 | G | . | . | # | . | . | # | . | . | . | . | . |
  +---+---+---+---+---+---+---+---+---+---+---+---+
4 | . | . | . | . | . | . | # | . | . | # | . | . |
  +---+---+---+---+---+---+---+---+---+---+---+---+
5 | . | . | V | . | # | # | # | . | . | # | . | . |
  +---+---+---+---+---+---+---+---+---+---+---+---+
6 | . | . | . | . | . | . | . | . | . | . | . | D |
  +---+---+---+---+---+---+---+---+---+---+---+---+

Legend:
C = Captain
G = Guard
S = Scout
V = Villager
# = blocked cell
D = destination

=== 

## Conclusiones

A lo largo de este notebook se mostró la construcción de una simulación de NPCs basada en componentes modulares.

NPCore permite integrar en un mismo sistema:

- toma de decisiones basada en reglas
- memoria de eventos
- emociones y personalidad
- aprendizaje a partir de resultados previos
- relaciones sociales y comportamiento grupal
- navegación en entornos con obstáculos y costos
- visualización de simulaciones en forma de cuadrícula

Esto demuestra que es posible generar comportamiento emergente a partir de reglas locales relativamente simples.

El proyecto no solo funciona como una librería para simulación, sino también como una base experimental para estudiar sistemas multiagente, inteligencia artificial para videojuegos y comportamiento autónomo en entornos dinámicos.

---

## Aportaciones del proyecto

Entre las principales aportaciones de NPCore se encuentran:

- una arquitectura modular y extensible
- separación clara entre `Brain`, `NPC` y `Environment`
- integración entre comportamiento individual y social
- soporte para navegación espacial y evaluación de riesgo
- capacidad de representar simulaciones de manera interpretable

Además, el uso de pruebas y ejemplos progresivos permite validar el sistema y facilitar su extensión futura.

---

## Trabajo futuro

Aunque el sistema ya es funcional, existen varias líneas de mejora que permitirían llevar NPCore a un nivel más avanzado:

- interfaz interactiva en notebooks con controles y sliders
- almacenamiento de simulaciones en JSON
- replay paso a paso de simulaciones
- animaciones con matplotlib
- generación automática de narrativas más elaboradas
- visualizaciones más avanzadas del entorno
- estrategias de decisión más complejas, como Utility AI
- aprendizaje adaptativo más profundo entre simulaciones

Estas extensiones convertirían a NPCore en una plataforma aún más sólida para experimentación, docencia y prototipos de inteligencia artificial.

---

## Cierre

NPCore muestra cómo combinar diseño modular, simulación y comportamiento emergente en una librería de Python clara y extensible.

El proyecto sirve como una demostración técnica de:

- modelado de agentes autónomos
- diseño orientado a objetos
- simulación multiagente
- integración de lógica, memoria, aprendizaje y entorno

En conjunto, este trabajo representa una base sólida para seguir construyendo sistemas más complejos e inteligentes.
